In [1]:
%%capture output
!pip install -U huggingface_hub
from huggingface_hub import hf_hub_download
from huggingface_hub import login, HfApi, notebook_login
import copy
import os
from collections import Counter

import cv2
import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
from PIL import Image, UnidentifiedImageError
from timm.models.layers import DropPath, trunc_normal_
from torch.utils.data import (DataLoader, Dataset, Subset,
                              WeightedRandomSampler)
from torchvision import datasets, transforms
from tqdm import tqdm

In [2]:
TRAIN_DIR = './StyleClassificationIndoors/StyleClassificationIndoors/train' # Update if your path is different
TEST_DIR = './StyleClassificationIndoors/StyleClassificationIndoors/test'
IMG_SIZE = 224
CLASSES = {
    'asian': 0, 'boho': 1, 'coastal': 2, 'contemporary': 3, 'craftsman': 4,
    'eclectic': 5, 'farmhouse': 6, 'french-country': 7, 'industrial': 8,
    'mediterranean': 9, 'minimalist': 10, 'modern': 11, 'scandinavian': 12,
    'shabby-chic-style': 13, 'southwestern': 14, 'tropical': 15, 'victorian': 16
}

In [3]:
def show_sample_images(directory):
    plt.figure(figsize=(15, 10))
    for i, style in enumerate(CLASSES.keys()):
        style_path = os.path.join(directory, style)
        img_name = os.listdir(style_path)[0]
        img_path = os.path.join(style_path, img_name)
        
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        plt.subplot(4, 5, i+1)
        plt.imshow(img)
        plt.title(style)
        plt.axis('off')
    plt.tight_layout()
    plt.show()

# show_sample_images(TRAIN_DIR)

In [ ]:
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.6, 1.0)), 
    transforms.RandomHorizontalFlip(),
    transforms.TrivialAugmentWide(),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD)
])

val_tf = transforms.Compose([
    transforms.Resize(IMG_SIZE + 32),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD)
])


In [5]:
class LayerNorm(nn.Module):
    r""" LayerNorm that supports two data formats: channels_last (default) or channels_first. 
    The ordering of the dimensions in the inputs. channels_last corresponds to inputs with 
    shape (batch_size, height, width, channels) while channels_first corresponds to inputs 
    with shape (batch_size, channels, height, width).
    """
    def __init__(self, normalized_shape, eps=1e-6, data_format="channels_last"):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(normalized_shape))
        self.bias = nn.Parameter(torch.zeros(normalized_shape))
        self.eps = eps
        self.data_format = data_format
        if self.data_format not in ["channels_last", "channels_first"]:
            raise NotImplementedError 
        self.normalized_shape = (normalized_shape, )
    
    def forward(self, x):
        if self.data_format == "channels_last":
            return F.layer_norm(x, self.normalized_shape, self.weight, self.bias, self.eps)
        elif self.data_format == "channels_first":
            u = x.mean(1, keepdim=True)
            s = (x - u).pow(2).mean(1, keepdim=True)
            x = (x - u) / torch.sqrt(s + self.eps)
            x = self.weight[:, None, None] * x + self.bias[:, None, None]
            return x

In [ ]:
class PatchEmbed(nn.Module):
    """ Split image into patches and embed them. """
    def __init__(self, img_size=224, patch_size=16, in_chans=3, embed_dim=768):
        super().__init__()
        self.img_size = img_size
        self.patch_size = patch_size
        self.grid_size = (img_size // patch_size, img_size // patch_size)
        self.num_patches = self.grid_size[0] * self.grid_size[1]

        self.proj = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        x = self.proj(x) # (B, Embed_Dim, Grid_H, Grid_W)
        x = x.flatten(2).transpose(1, 2) # (B, Num_Patches, Embed_Dim)
        return x

In [16]:
class Attention(nn.Module):
    def __init__(self, dim, num_heads=8, qkv_bias=False, attn_drop=0., proj_drop=0.):
        super().__init__()
        self.num_heads = num_heads
        head_dim = dim // num_heads
        self.scale = head_dim ** -0.5

        self.qkv = nn.Linear(dim, dim * 3, bias=qkv_bias)
        self.attn_drop = nn.Dropout(attn_drop)
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(proj_drop)

    def forward(self, x):
        B, N, C = x.shape
        # (B, N, 3, Heads, Head_Dim) -> (3, B, Heads, N, Head_Dim)
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]

        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        attn = self.attn_drop(attn)

        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        x = self.proj(x)
        x = self.proj_drop(x)
        return x

In [17]:
class Mlp(nn.Module):
    def __init__(self, in_features, hidden_features=None, out_features=None, drop=0.):
        super().__init__()
        out_features = out_features or in_features
        hidden_features = hidden_features or in_features
        self.fc1 = nn.Linear(in_features, hidden_features)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(hidden_features, out_features)
        self.drop = nn.Dropout(drop)

    def forward(self, x):
        x = self.fc1(x)
        x = self.act(x)
        x = self.drop(x)
        x = self.fc2(x)
        x = self.drop(x)
        return x

In [18]:
class Block(nn.Module):
    def __init__(self, dim, num_heads, mlp_ratio=4., qkv_bias=False, drop=0., attn_drop=0.):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim, eps=1e-6)
        self.attn = Attention(dim, num_heads=num_heads, qkv_bias=qkv_bias, attn_drop=attn_drop, proj_drop=drop)
        self.norm2 = nn.LayerNorm(dim, eps=1e-6)
        self.mlp = Mlp(in_features=dim, hidden_features=int(dim * mlp_ratio), drop=drop)

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x

In [ ]:
class VisionTransformer(nn.Module):
    def __init__(self, img_size=224, patch_size=16, in_chans=3, num_classes=1000, 
                 embed_dim=768, depth=12, num_heads=12, mlp_ratio=4., qkv_bias=True, drop_rate=0.):
        super().__init__()
        self.num_classes = num_classes
        self.embed_dim = embed_dim
        
        self.patch_embed = PatchEmbed(img_size, patch_size, in_chans, embed_dim)
        num_patches = self.patch_embed.num_patches

        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches + 1, embed_dim))
        self.pos_drop = nn.Dropout(p=drop_rate)

        self.blocks = nn.Sequential(*[
            Block(dim=embed_dim, num_heads=num_heads, mlp_ratio=mlp_ratio, qkv_bias=qkv_bias, drop=drop_rate)
            for i in range(depth)])

        self.norm = nn.LayerNorm(embed_dim, eps=1e-6)
        self.head = nn.Linear(embed_dim, num_classes) if num_classes > 0 else nn.Identity()

        nn.init.trunc_normal_(self.pos_embed, std=.02)
        nn.init.trunc_normal_(self.cls_token, std=.02)
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.trunc_normal_(m.weight, std=.02)
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.LayerNorm):
            nn.init.constant_(m.bias, 0)
            nn.init.constant_(m.weight, 1.0)

    def forward_features(self, x):
        B = x.shape[0]
        x = self.patch_embed(x)
        
        cls_tokens = self.cls_token.expand(B, -1, -1) 
        x = torch.cat((cls_tokens, x), dim=1)
        x = x + self.pos_embed
        x = self.pos_drop(x)

        x = self.blocks(x)
        x = self.norm(x)
        return x[:, 0] 

    def forward(self, x):
        x = self.forward_features(x)
        x = self.head(x)
        return x

In [ ]:
def get_pretrained_vit_large(num_classes_target=17):
    cfg = {
        "img_size": 224, 
        "patch_size": 16, 
        "embed_dim": 1024,   
        "depth": 24,         
        "num_heads": 16,     
        "num_classes": 21843, 
        "url": "https://github.com/rwightman/pytorch-image-models/releases/download/v0.1-vitjx/jx_vit_large_patch16_224_in21k-606da67d.pth"
    }
    
    # 1. Initialize Architecture
    model = VisionTransformer(
        img_size=cfg['img_size'], 
        patch_size=cfg['patch_size'],
        embed_dim=cfg['embed_dim'], 
        depth=cfg['depth'], 
        num_heads=cfg['num_heads'], 
        num_classes=cfg['num_classes']
    )

    print("Downloading ViT-Large/16 weights (approx 1.2GB)...")
    checkpoint = torch.hub.load_state_dict_from_url(cfg["url"], map_location="cpu", progress=True)
    
    if 'state_dict' in checkpoint:
        state_dict = checkpoint['state_dict']
    else:
        state_dict = checkpoint

    new_state_dict = {}
    for k, v in state_dict.items():
        name = k.replace("module.", "") 
        new_state_dict[name] = v
        
    model.load_state_dict(new_state_dict, strict=False)
    print("Large weights loaded successfully.")

    model.head = nn.Linear(cfg['embed_dim'], num_classes_target)
    
    nn.init.trunc_normal_(model.head.weight, std=.02)
    nn.init.constant_(model.head.bias, 0)
    
    return model

In [ ]:
full_dataset = datasets.ImageFolder(root=TRAIN_DIR, transform=train_tf)

targets = full_dataset.targets
class_counts = Counter(targets)
class_weights = {cls: 1.0/count for cls, count in class_counts.items()}
sample_weights = [class_weights[t] for t in targets]
sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

full_loader = DataLoader(
    full_dataset,
    batch_size=16,         
    sampler=sampler,
    num_workers=2,
    pin_memory=True
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = get_pretrained_vit_large(num_classes_target=17)
model = model.to(device)

if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs")
    model = nn.DataParallel(model)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5, weight_decay=0.1)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=12) # 12 Epochs
criterion = nn.CrossEntropyLoss()

In [ ]:
def train_full_data(model, loader, criterion, optimizer, scheduler, device, epochs=12):
    print(f"Starting training on {len(loader.dataset)} images for {epochs} epochs.")
    
    for epoch in range(epochs):
        print(f"\nEpoch {epoch+1}/{epochs}")
        model.train()
        
        running_loss = 0.0
        correct = 0
        total = 0
        
        loop = tqdm(loader, leave=True)
        
        for images, labels in loop:
            images, labels = images.to(device), labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            optimizer.zero_grad()
            loss.backward()
            
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            
            optimizer.step()
            
            # Metrics
            running_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
            loop.set_description(f"Loss: {loss.item():.4f}")
            
        epoch_loss = running_loss / len(loader.dataset)
        epoch_acc = correct / total
        
        print(f"Train Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}")
        
        scheduler.step()

    save_path = "vit_large_full_data.pth"
    torch.save(model.state_dict(), save_path)
    print(f"Training Complete. Model saved to {save_path}")
    return model

model = train_full_data(
    model, 
    full_loader, 
    criterion, 
    optimizer, 
    scheduler, 
    device, 
    epochs=15
)


In [ ]:
notebook_login()

In [ ]:
api = HfApi()

api.create_repo(repo_id="mostafafaheem/ViT-large", exist_ok=True)

api.upload_file(
    path_or_fileobj="/kaggle/working/vit_large_full_data.pth",
    path_in_repo="model.pth",
    repo_id="mostafafaheem/ViT-large",
    repo_type="model"
)

In [ ]:
def load_vit_from_hub(model_type="large", num_classes=17, repo_id="mostafafaheem/ViT-large", filename="model.pth"):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    if model_type == "large":
        cfg = {"embed_dim": 1024, "depth": 24, "num_heads": 16}
    else:
        cfg = {"embed_dim": 768, "depth": 12, "num_heads": 12}
        
    print(f"Initializing ViT-{model_type.capitalize()}...")
    model = VisionTransformer(
        img_size=224, 
        patch_size=16,
        embed_dim=cfg['embed_dim'], 
        depth=cfg['depth'], 
        num_heads=cfg['num_heads'], 
        num_classes=num_classes
    )
    
    print(f"Downloading weights from {repo_id}...")
    try:
        model_path = hf_hub_download(repo_id=repo_id, filename=filename, repo_type="model")
    except Exception as e:
        print("Error downloading: Ensure your Repo ID and Filename are correct.")
        raise e

    state_dict = torch.load(model_path, map_location="cpu")
    new_state_dict = {}
    for k, v in state_dict.items():
        name = k.replace("module.", "")
        new_state_dict[name] = v
        
    model.load_state_dict(new_state_dict)
    model.to(device)
    model.eval()
    print("Model Loaded Successfully!")
    return model, device

In [ ]:
test_tf = transforms.Compose([
    transforms.Resize((224, 224)), 
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]) 
])

class TestDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.filenames = [f for f in os.listdir(root_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        filename = self.filenames[idx]
        img_path = os.path.join(self.root_dir, filename)
        
        try:
            image = Image.open(img_path).convert('RGB')
        except (UnidentifiedImageError, OSError):
            print(f"Warning: Corrupted image found: {filename}. Using dummy input.")

            image = Image.new('RGB', (IMG_SIZE, IMG_SIZE), color='black')
        
        if self.transform:
            image = self.transform(image)        
            
        return image, filename

In [ ]:
REPO_ID = "mostafafaheem/ViT-large" 
FILENAME = "model.pth"   
MODEL_TYPE = "large"                   
NUM_CLASSES = 17

model, device = load_vit_from_hub(MODEL_TYPE, NUM_CLASSES, REPO_ID, FILENAME)

test_dataset = TestDataset(TEST_DIR, transform=test_tf)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)

all_filenames = []
all_predictions = []

print(f"Starting inference on {len(test_dataset)} images...")
with torch.no_grad():
    for images, filenames in tqdm(test_loader):
        images = images.to(device)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        
        all_filenames.extend(filenames)
        all_predictions.extend(preds.cpu().numpy())

df = pd.DataFrame({
    'ImageName': all_filenames,    
    'ClassLabel': all_predictions 
})
df.to_csv('submission.csv', index=False)
print("Submission saved to submission.csv")

Initializing ViT-Large...


model.pth:   0%|          | 0.00/1.21G [00:00<?, ?B/s]

Model Loaded Successfully!
Starting inference on 5482 images...


  1%|          | 1/172 [00:02<07:47,  2.73s/it]/usr/local/lib/python3.11/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
 29%|██▉       | 50/172 [00:56<02:20,  1.15s/it]

 30%|███       | 52/172 [00:58<02:19,  1.16s/it]/usr/local/lib/python3.11/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
 33%|███▎      | 56/172 [01:03<02:16,  1.18s/it]

 46%|████▌     | 79/172 [01:32<01:58,  1.27s/it]

 76%|███████▌  | 130/172 [02:41<00:54,  1.31s/it]

100%|██████████| 172/172 [03:36<00:00,  1.26s/it]

Submission saved to submission.csv
